In [0]:
#loading data:-
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Loading Data").getOrCreate()
df_delays=spark.read.csv("/databricks-datasets/flights/departuredelays.csv",header=True,inferSchema=True)
df_delays.show()

+-------+-----+--------+------+-----------+
|   date|delay|distance|origin|destination|
+-------+-----+--------+------+-----------+
|1011245|    6|     602|   ABE|        ATL|
|1020600|   -8|     369|   ABE|        DTW|
|1021245|   -2|     602|   ABE|        ATL|
|1020605|   -4|     602|   ABE|        ATL|
|1031245|   -4|     602|   ABE|        ATL|
|1030605|    0|     602|   ABE|        ATL|
|1041243|   10|     602|   ABE|        ATL|
|1040605|   28|     602|   ABE|        ATL|
|1051245|   88|     602|   ABE|        ATL|
|1050605|    9|     602|   ABE|        ATL|
|1061215|   -6|     602|   ABE|        ATL|
|1061725|   69|     602|   ABE|        ATL|
|1061230|    0|     369|   ABE|        DTW|
|1060625|   -3|     602|   ABE|        ATL|
|1070600|    0|     369|   ABE|        DTW|
|1071725|    0|     602|   ABE|        ATL|
|1071230|    0|     369|   ABE|        DTW|
|1070625|    0|     602|   ABE|        ATL|
|1071219|    0|     569|   ABE|        ORD|
|1080600|    0|     369|   ABE| 

In [0]:
df_delays.count()

1391578

In [0]:
df_delays=df_delays.orderBy("date")
df_delays.show()

+-------+-----+--------+------+-----------+
|   date|delay|distance|origin|destination|
+-------+-----+--------+------+-----------+
|1010005|   -8|    2024|   LAX|        PBI|
|1010010|   -6|    1980|   SEA|        CLT|
|1010020|    0|    1273|   SFO|        DFW|
|1010020|   -2|    1995|   SFO|        CLT|
|1010023|   14|    1421|   SFO|        IAH|
|1010025|   -3|    1452|   PHX|        DTW|
|1010025|   33|    1198|   LAX|        IAH|
|1010029|   49|    1061|   LAS|        IAH|
|1010030|   -2|    1983|   PDX|        CLT|
|1010030|   -7|    2191|   SFO|        PHL|
|1010030|   -8|    1518|   LAS|        ATL|
|1010035|   -5|    1846|   LAX|        CLT|
|1010035|   -1|    1259|   ANC|        SEA|
|1010040|   -6|    1382|   SLC|        ATL|
|1010043|   18|    1413|   DEN|        JFK|
|1010045|  -11|    1891|   LAS|        PHL|
|1010050|   -6|    1340|   ANC|        PDX|
|1010053|   14|    1259|   ANC|        SEA|
|1010055|   -2|    2087|   LAX|        PHL|
|1010059|   -9|    1042|   DEN| 

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,col
window_spec=Window.partitionBy().orderBy("date","origin")
df_delays.withColumn("RowNum",row_number().over(window_spec)).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------+-----+--------+------+-----------+------+
|   date|delay|distance|origin|destination|RowNum|
+-------+-----+--------+------+-----------+------+
|1010005|   -8|    2024|   LAX|        PBI|     1|
|1010010|   -6|    1980|   SEA|        CLT|     2|
|1010020|   -2|    1995|   SFO|        CLT|     3|
|1010020|    0|    1273|   SFO|        DFW|     4|
|1010023|   14|    1421|   SFO|        IAH|     5|
|1010025|   33|    1198|   LAX|        IAH|     6|
|1010025|   -3|    1452|   PHX|        DTW|     7|
|1010029|   49|    1061|   LAS|        IAH|     8|
|1010030|   -8|    1518|   LAS|        ATL|     9|
|1010030|   -2|    1983|   PDX|        CLT|    10|
|1010030|   -7|    2191|   SFO|        PHL|    11|
|1010035|   -1|    1259|   ANC|        SEA|    12|
|1010035|   -5|    1846|   LAX|        CLT|    13|
|1010040|   -6|    1382|   SLC|        ATL|    14|
|1010043|   18|    1413|   DEN|        JFK|    15|
|1010045|  -11|    1891|   LAS|        PHL|    16|
|1010050|   -6|    1340|   ANC|

In [0]:
df_delays_pd1=df_delays.withColumn("RowNum",row_number().over(window_spec)).filter(col("RowNum") < 1001).toPandas()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_delays_pd1.shape

(1000, 6)

In [0]:
df_delays_pd1.to_csv("/Volumes/main/autoloader/bronze/raw/data_file1.csv",header=True)

In [0]:
df_delays_pd2=df_delays.withColumn("RowNum",row_number().over(window_spec)).filter((col("RowNum") > 1000) &(col("RowNum") < 2001)).toPandas()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_delays_pd2.head()

,date,delay,distance,origin,destination,RowNum
0,1010705,-4,307,LGB,SFO,1001
1,1010705,-2,400,MCI,DFW,1002
2,1010705,24,1255,MDW,PHX,1003
3,1010705,-7,827,MHT,ATL,1004
4,1010705,-2,2035,MIA,LAX,1005


In [0]:
df_delays_pd2.to_csv("/Volumes/main/autoloader/bronze/raw/data_file2.csv",header=True,index=False)